In [2]:
import torch
from torch import nn
from d2l import torch as d2l

In [11]:
batch_size = 256

train_iter,test_iter = d2l.load_data_fashion_mnist(batch_size)

net = nn.Sequential(nn.Flatten(),nn.Linear(784,10))

def init_weights(m):
    if type(m)==nn.Linear:
        nn.init.normal_(m.weight,std=0.01) #_表示原地操作

net.apply(init_weights)

loss = nn.CrossEntropyLoss()

trainer = torch.optim.SGD(net.parameters(),lr=0.1)

epochs = 10

def train_ch3(epochs,net,loss,updater,train_iter,test_iter):
    for epoch in range(epochs):
        l,a = train_epoch_ch3(net,train_iter,loss,updater)
        print(f"epoch:{epoch} \t loss:{l:.3f} \t acc:{a:.3f}\n")

def accuracy(y_hat,y):
    y_hat = y_hat.argmax(axis=1)
    if len(y_hat) == len(y):
        y_a = y_hat.type(y.dtype)==y
    return float(y_a.type(y.dtype).sum())

def train_epoch_ch3(net,train_iter,loss,updater):
    if isinstance(net,torch.nn.Module):
        net.train()
    metric=Accumulator(3)
    for X,y in train_iter:
        y_hat = net(X)
        l = loss(y_hat,y)
        if isinstance(updater,torch.optim.Optimizer):
            updater.zero_grad()
            l.backward()
            updater.step()
            metric.add(float(l)*len(y),accuracy(y_hat,y),y.numel())
        else:
            l.sum().backward()
            updater(X.shape[0])
            metric.add(float(l.sum()),accuracy(y_hat,y),y.numel())

    return metric[0]/metric[2],metric[1]/metric[2]
class Accumulator:
    def __init__(self,n):
        self.data = [0.0]*n

    def add(self,*args):
        self.data = [a+float(b) for a,b in zip(self.data,args)]

    def __getitem__(self, idx):
        return self.data[idx]

    def reset(self):
        self.data = [0.0]*len(self.data)


In [12]:
train_ch3(epochs,net,loss,trainer,train_iter,test_iter)

epoch:0 	 loss:0.786 	 acc:0.749

epoch:1 	 loss:0.570 	 acc:0.812

epoch:2 	 loss:0.525 	 acc:0.824

epoch:3 	 loss:0.501 	 acc:0.832

epoch:4 	 loss:0.485 	 acc:0.837

epoch:5 	 loss:0.475 	 acc:0.840

epoch:6 	 loss:0.465 	 acc:0.844

epoch:7 	 loss:0.458 	 acc:0.845

epoch:8 	 loss:0.453 	 acc:0.846

epoch:9 	 loss:0.448 	 acc:0.848

